# Neural Network Model Testing

# 1. Importing Libraries and the Data

In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from tensorflow import keras
import matplotlib.pyplot as plt
from collect_log_data import read_log_data

In [11]:
inputs, targets = read_log_data(verbose = True)

Reading prepared data from csv files...
Prepared data collected


# 2. Testing a Basic Neural Network

In [86]:
nn = keras.models.Sequential(name = "Basic_Neural_Network")

nn.add(keras.layers.Input(shape = (inputs.shape[1],)))
nn.add(keras.layers.Dense(units = 15, activation = 'relu', name = "Hidden_Layer"))
nn.add(keras.layers.Dense(units = targets.shape[1], activation = 'sigmoid', name = "Output_Layer"))

optimizer = keras.optimizers.Adam()
loss_function = 'categorical_crossentropy'

nn.compile(optimizer = optimizer, loss = loss_function)
nn.summary()

Model: "Basic_Neural_Network"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Hidden_Layer (Dense)            │ (None, 15)             │           285 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output_Layer (Dense)            │ (None, 12)             │           192 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 477 (1.86 KB)

 Trainable params: 477 (1.86 KB)

 Non-trainable params: 0 (0.00 B)

In [87]:
train_inputs, test_inputs, train_targets, test_targets = train_test_split(inputs, targets, test_size = 0.1)

In [90]:
nn.fit(train_inputs, train_targets, batch_size = 32, epochs = 100, verbose = False)

In [96]:
def match_predictor_model_evaluation(test_inputs, test_targets):
	test_predictions = nn.predict(test_inputs)

	test_predictions_split = np.hsplit(test_predictions, 2)
	red_score_predictions = np.argmax(test_predictions_split[0], axis = -1)
	blu_score_predictions = np.argmax(test_predictions_split[1], axis = -1)

	expected_scores = np.hsplit(test_targets, 2)
	red_score_targets = np.argmax(expected_scores[0], axis = -1)
	blu_score_targets = np.argmax(expected_scores[1], axis = -1)

	# Original, previous evaluation
	exact_score_accuracy = sum(np.hstack((red_score_predictions, blu_score_predictions)) ==\
									np.hstack((red_score_targets, blu_score_targets))) / test_targets.shape[0] * 100

	mae = mean_absolute_error(np.hstack((red_score_targets, blu_score_targets)),\
								np.hstack((red_score_predictions, red_score_predictions)))

	correct_winners = 0
	for i in range(test_targets.shape[0]):

		if red_score_predictions[i] > blu_score_predictions[i]:
			predicted_winner = 0
		elif red_score_predictions[i] < blu_score_predictions[i]:
			predicted_winner = 1
		else:
			predicted_winner = 2 # tie
		
		if red_score_targets[i] > blu_score_targets[i]:
			actual_winner = 0
		elif red_score_targets[i] < blu_score_targets[i]:
			actual_winner = 1
		else:
			actual_winner = 2 # tie
		
		if predicted_winner == actual_winner:
			correct_winners += 1

	correct_winner_accuracy = correct_winners / test_targets.shape[0] * 100
	return exact_score_accuracy, mae, correct_winner_accuracy

In [97]:
exact_score_accuracy, mae, correct_winner_accuracy = match_predictor_model_evaluation(test_inputs, test_targets)

print(f"Exact Score Accuracy: {exact_score_accuracy:.2f}%")
print(f"MAE: {mae:.2f}")
print(f"Correct Winner Accuracy: {correct_winner_accuracy:.2f}%")

75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 456us/step
Exact Score Accuracy: 32.74%
MAE: 2.42
Correct Winner Accuracy: 5.63%


The model successfully trained and predicted, but it's scores are very low. After training for 100 epochs, it only guessed the exact correct score of each team approximately 1 in 3 times (approximately the same result on average as the last version of this neural network) and it only guessed the winning team correctly 5% of the time, which is extremely strange since it guesses the correct scores exactly more often than that.

To begin determining the cause, the predictions it made are printed below so they can be examined.

In [98]:
test_predictions = nn.predict(test_inputs)
print(test_predictions)

75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 487us/step
[[1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]
 ...
 [1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]]


It appears to be predicting a 100% chance of every score strangely. This can be verified by examining the sum of all its predictions, which if true, should add up to 0. This is printed below.

In [99]:
test_predictions_split = np.hsplit(test_predictions, 2)
red_score_predictions = np.argmax(test_predictions_split[0], axis = -1)
blu_score_predictions = np.argmax(test_predictions_split[1], axis = -1)
print(sum(red_score_predictions))
print(sum(blu_score_predictions))

0
0


This model does in fact predict a 100% chance for every score all the time. It could be caused by the model having its predictions be swayed all over the place too much by the random starting weights of each layer. A smaller learning rate may fix this issue. This is tested below.

In [103]:
nn = keras.models.Sequential(name = "Basic_Neural_Network")

nn.add(keras.layers.Input(shape = (inputs.shape[1],)))
nn.add(keras.layers.Dense(units = 15, activation = 'relu', name = "Hidden_Layer"))
nn.add(keras.layers.Dense(units = targets.shape[1], activation = 'sigmoid', name = "Output_Layer"))

optimizer = keras.optimizers.Adam(learning_rate = 0.00001) # Decrease learning rate from default of 0.001
loss_function = 'categorical_crossentropy'

nn.compile(optimizer = optimizer, loss = loss_function)
nn.summary()

Model: "Basic_Neural_Network"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Hidden_Layer (Dense)            │ (None, 15)             │           285 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output_Layer (Dense)            │ (None, 12)             │           192 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 477 (1.86 KB)

 Trainable params: 477 (1.86 KB)

 Non-trainable params: 0 (0.00 B)

In [104]:
nn.fit(train_inputs, train_targets, batch_size = 32, epochs = 100, verbose = False)

In [106]:
exact_score_accuracy, mae, correct_winner_accuracy = match_predictor_model_evaluation(test_inputs, test_targets)

print(f"Exact Score Accuracy: {exact_score_accuracy:.2f}%")
print(f"MAE: {mae:.2f}")
print(f"Correct Winner Accuracy: {correct_winner_accuracy:.2f}%")

test_predictions = nn.predict(test_inputs)
print(test_predictions)

test_predictions_split = np.hsplit(test_predictions, 2)
red_score_predictions = np.argmax(test_predictions_split[0], axis = -1)
blu_score_predictions = np.argmax(test_predictions_split[1], axis = -1)
print(sum(red_score_predictions))
print(sum(blu_score_predictions))

75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 473us/step
Exact Score Accuracy: 33.15%
MAE: 2.42
Correct Winner Accuracy: 20.18%
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 478us/step
[[0.99999994 0.99999994 0.9999998  ... 0.99999994 0.99999994 0.99999994]
 [1.         0.99999994 0.99999994 ... 1.         0.99999994 1.        ]
 [0.99999994 0.99999994 0.9999998  ... 0.99999994 0.99999994 1.        ]
 ...
 [1.         1.         1.         ... 1.         1.         1.        ]
 [1.         0.99999994 0.9999999  ... 1.         0.99999994 0.99999994]
 [1.         1.         1.         ... 1.         1.         1.        ]]
32
1509


A smaller learning rate does seem to positively impact the correct winner accuracy in addition to lowering the category confidence down from always being 100% for some values.

# 2. Testing for Better Learning Rates

Below, various learning rates are tested to search for which one can get the best results in 100 epochs of training. The learning rates start at the one tested above and decrease by an order of magnitude each iteration.

In [109]:
learning_rate = 0.00001
for _ in range(10):
	nn = keras.models.Sequential(name = "Basic_Neural_Network")

	nn.add(keras.layers.Input(shape = (inputs.shape[1],)))
	nn.add(keras.layers.Dense(units = 15, activation = 'relu', name = "Hidden_Layer"))
	nn.add(keras.layers.Dense(units = targets.shape[1], activation = 'sigmoid', name = "Output_Layer"))

	optimizer = keras.optimizers.Adam(learning_rate = 0.0000000001)
	loss_function = 'categorical_crossentropy'

	nn.compile(optimizer = optimizer, loss = loss_function)

	nn.fit(train_inputs, train_targets, batch_size = 32, epochs = 100, verbose = False)

	exact_score_accuracy, mae, correct_winner_accuracy = match_predictor_model_evaluation(test_inputs, test_targets)
	print(f"Learning Rate: {learning_rate}")
	print(f"Exact Score Accuracy: {exact_score_accuracy:.2f}%")
	print(f"MAE: {mae:.2f}")
	print(f"Correct Winner Accuracy: {correct_winner_accuracy:.2f}%")

	learning_rate /= 10

75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 603us/step
Learning Rate: 1e-05
Exact Score Accuracy: 33.32%
MAE: 1.74
Correct Winner Accuracy: 45.91%
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 610us/step
Learning Rate: 1.0000000000000002e-06
Exact Score Accuracy: 34.57%
MAE: 2.03
Correct Winner Accuracy: 47.50%
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 582us/step
Learning Rate: 1.0000000000000002e-07
Exact Score Accuracy: 32.61%
MAE: 1.88
Correct Winner Accuracy: 45.66%
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 587us/step
Learning Rate: 1.0000000000000002e-08
Exact Score Accuracy: 36.36%
MAE: 1.75
Correct Winner Accuracy: 15.89%
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 609us/step
Learning Rate: 1.0000000000000003e-09
Exact Score Accuracy: 33.44%
MAE: 1.77
Correct Winner Accuracy: 48.46%
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 593us/step
Learning Rate: 1.0000000000000003e-10
Exact Score Accuracy: 32.74%
MAE: 2.08
Correct Winner Accuracy: 38.49%
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 613us/step
Learning Rate: 1.0000000000000003e-11
Exact Score Accuracy: 35.82%
MAE: 1.78

A learning rate of 0.00000000001 got the best accuracy at guessing the winner, but even then it was only correct ~50% of the time. Additionally, some of the smallest learning rates, 1e-12 and 1e-14, had the lowest MAEs. This suggests that an extremely small learning rate may be the best if the model it's used on is given even more epochs to train on. Below, a model with a learning rate of 1e-20 is tested over 1,000 epochs.

In [110]:
nn = keras.models.Sequential(name = "Basic_Neural_Network")

nn.add(keras.layers.Input(shape = (inputs.shape[1],)))
nn.add(keras.layers.Dense(units = 15, activation = 'relu', name = "Hidden_Layer"))
nn.add(keras.layers.Dense(units = targets.shape[1], activation = 'sigmoid', name = "Output_Layer"))

optimizer = keras.optimizers.Adam(learning_rate = 1e-20)
loss_function = 'categorical_crossentropy'

nn.compile(optimizer = optimizer, loss = loss_function)
nn.summary()

Model: "Basic_Neural_Network"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Hidden_Layer (Dense)            │ (None, 15)             │           285 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output_Layer (Dense)            │ (None, 12)             │           192 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 477 (1.86 KB)

 Trainable params: 477 (1.86 KB)

 Non-trainable params: 0 (0.00 B)

In [112]:
nn.fit(train_inputs, train_targets, batch_size = 32, epochs = 1000, verbose = False)

In [113]:
exact_score_accuracy, mae, correct_winner_accuracy = match_predictor_model_evaluation(test_inputs, test_targets)
print(f"Exact Score Accuracy: {exact_score_accuracy:.2f}%")
print(f"MAE: {mae:.2f}")
print(f"Correct Winner Accuracy: {correct_winner_accuracy:.2f}%")

75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 705us/step
Exact Score Accuracy: 30.53%
MAE: 2.11
Correct Winner Accuracy: 45.29%


The hypothesis of a smaller learning rate doing better given more epochs to train on does not seem to be the case. Instead, this suggests that the low MAE for smaller learning rates in the previous tests was luck from random starting weights. Instead of experimenting with the learning rates more, other hyperparameters should now be tested. Carrying forward, the learning rate that had the best winning team accuracy of 1e-11 will be used.

In [114]:
learning_rate = 1e-11
# learning_rate = 1e-10 # had good results with this from several other tests

# 3. Testing for Better Hidden Neuron Counts and Layer Counts

Below is code to iterate through various numbers of hidden layers and neurons in each hidden layer to test each of them in order to find the best combination. Since this test will likely take hours, the logic of the loop for this test was first tested in `hyperparameter_loop_test.py`.

In [ ]:
def print_model_details(model_details):
	print("Model:")
	print(f"Hidden Layers: {model_details[0]}, Hidden Neuron Counts: {model_details[1]}")

min_neurons = 10
max_neurons = 20

best_exact_score = 0
best_mae = float('inf')
best_winner_accuracy = 0

best_exact_score_model = None
best_mae_model = None
best_winner_accuracy_model = None

for hidden_layer_count in range(1, 4):
	hidden_neuron_counts = [min_neurons] * hidden_layer_count
	outer_layer = 0
	print(f"-------------------- Hidden Layers: {hidden_layer_count}")
	while outer_layer < hidden_layer_count:
		nn = keras.models.Sequential(name = "Basic_Neural_Network")

		nn.add(keras.layers.Input(shape = (inputs.shape[1],)))
		for i in range(hidden_layer_count):
			nn.add(keras.layers.Dense(units = hidden_neuron_counts[i], activation = 'relu', name = f"Hidden_Layer_{i+1}"))
		nn.add(keras.layers.Dense(units = targets.shape[1], activation = 'sigmoid', name = "Output_Layer"))

		optimizer = keras.optimizers.Adam(learning_rate = learning_rate)
		loss_function = 'categorical_crossentropy'

		nn.compile(optimizer = optimizer, loss = loss_function)

		nn.fit(train_inputs, train_targets, batch_size = 32, epochs = 50, verbose = False)

		exact_score_accuracy, mae, correct_winner_accuracy = match_predictor_model_evaluation(test_inputs, test_targets)
		print(f"Hidden Neuron Counts: {hidden_neuron_counts} (Hidden Layers: {hidden_layer_count})")
		print(f"Exact Score Accuracy: {exact_score_accuracy:.2f}%")
		print(f"MAE: {mae:.2f}")
		print(f"Correct Winner Accuracy: {correct_winner_accuracy:.2f}%")

		if exact_score_accuracy > best_exact_score:
			best_exact_score = exact_score_accuracy
			best_exact_score_model = (hidden_layer_count, hidden_neuron_counts.copy())
			print(f"New best Exact Score: {best_exact_score}")
			print_model_details(best_exact_score_model)

		if mae < best_mae:
			best_mae = mae
			best_mae_model = (hidden_layer_count, hidden_neuron_counts.copy())
			print(f"New best MAE: {best_mae}")
			print_model_details(best_mae_model)

		if correct_winner_accuracy > best_winner_accuracy:
			best_winner_accuracy = correct_winner_accuracy
			best_winner_accuracy_model = (hidden_layer_count, hidden_neuron_counts.copy())
			print(f"New best Winner Accuracy: {best_winner_accuracy}")
			print_model_details(best_winner_accuracy_model)

		current_layer = 0
		for i in range(hidden_layer_count):
			hidden_neuron_counts[i] += 1
			if hidden_neuron_counts[i] > max_neurons:
				hidden_neuron_counts[i] = min_neurons
				current_layer += 1
			else:
				break
		
		outer_layer = max(outer_layer, current_layer)

print(f"Best Exact Score: {best_exact_score}")
print("Model:")
print(f"Hidden Layers: {best_exact_score_model[0]}, Hidden Neuron Counts: {best_exact_score_model[1]}")

print(f"Best MAE: {best_mae}")
print("Model:")
print(f"Hidden Layers: {best_mae_model[0]}, Hidden Neuron Counts: {best_mae_model[1]}")

print(f"Best Winner Accuracy: {best_winner_accuracy}")
print("Model:")
print(f"Hidden Layers: {best_winner_accuracy_model[0]}, Hidden Neuron Counts: {best_winner_accuracy_model[1]}")

-------------------- Hidden Layers: 1
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 594us/step
Hidden Neuron Counts: [10] (Hidden Layers: 1)
Exact Score Accuracy: 35.86%
MAE: 1.90
Correct Winner Accuracy: 36.53%
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 744us/step
Hidden Neuron Counts: [11] (Hidden Layers: 1)
Exact Score Accuracy: 27.31%
MAE: 1.93
Correct Winner Accuracy: 43.04%
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 743us/step
Hidden Neuron Counts: [12] (Hidden Layers: 1)
Exact Score Accuracy: 28.48%
MAE: 1.96
Correct Winner Accuracy: 36.78%
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 743us/step
Hidden Neuron Counts: [13] (Hidden Layers: 1)
Exact Score Accuracy: 35.65%
MAE: 1.61
Correct Winner Accuracy: 35.07%
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 606us/step
Hidden Neuron Counts: [14] (Hidden Layers: 1)
Exact Score Accuracy: 37.07%
MAE: 1.75
Correct Winner Accuracy: 49.75%
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 607us/step
Hidden Neuron Counts: [15] (Hidden Layers: 1)
Exact Score Accuracy: 32.11%
MAE: 1.73
Correct Winner Accuracy: 36.16%
75/75 ━━━━━━━━

KeyboardInterrupt: 